In [ ]:
!pip install "transformers<5" tf-keras

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

In [ ]:
import tensorflow as tf
import tf_keras as keras
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordLevelTrainer
from transformers import PreTrainedTokenizerFast,GPT2Config,TFAutoModelForCausalLM,DataCollatorForLanguageModeling,TFGPT2LMHeadModel

In [ ]:
import numpy as np

In [ ]:
from datasets import Dataset
np.set_printoptions(precision=3,suppress=True)
tf.random.set_seed(42)

In [ ]:
sentences=[
    "a cat sat on the mat.",
    "the dog sat on the floor.",
    "a cat chased the dog .",
    "the mat is on the floor.",
    "the dog isun beleiv able"
]

In [ ]:
backend_tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
backend_tokenizer.pre_tokenizer = Whitespace()
trainer = WordLevelTrainer(special_tokens=["[UNK]","[PAD]"])
backend_tokenizer.train_from_iterator(sentences,trainer=trainer)
tokenizer = PreTrainedTokenizerFast(tokenizer_object=backend_tokenizer,pad_token="[PAD]",unk_token="[UNK]")

In [ ]:
text = "a cat sat on the mat ."
tokens = tokenizer.tokenize(text)
tokens

['a', 'cat', 'sat', 'on', 'the', 'mat', '.']

In [ ]:
print("vocaublary")
for word,idx in sorted(tokenizer.get_vocab().items(),key=lambda x:x[1]):
    print(f"{idx:2} : {word}")

vocaublary
 0 : [UNK]
 1 : [PAD]
 2 : the
 3 : .
 4 : dog
 5 : on
 6 : a
 7 : cat
 8 : floor
 9 : mat
10 : sat
11 : able
12 : beleiv
13 : chased
14 : is
15 : isun


In [ ]:
text = "a cat sat on the"
encoded = tokenizer(text,return_tensors="tf")
print("text")
print(text)
print("tokens")
print(tokenizer.tokenize(text))
print("token ids")
print(encoded["input_ids"].numpy())


TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


text
a cat sat on the
tokens
['a', 'cat', 'sat', 'on', 'the']
token ids
[[ 6  7 10  5  2]]


In [ ]:
ids = encoded["input_ids"][0]
one_hot = tf.one_hot(ids,depth=tokenizer.vocab_size)
print(one_hot.numpy().astype(int))

[[0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]]


In [ ]:
config = GPT2Config(
    vocab_size = tokenizer.vocab_size,
    n_positions=32,
    n_embd=32,
    n_layer=2,
    n_head=4,
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
    pad_token_id=tokenizer.pad_token_id,
    use_cache=False,
    output_hidden_states=True,
    output_attentions=True
)

In [ ]:
model = TFGPT2LMHeadModel(config)

The following generation flags are not valid and may be ignored: ['output_attentions', 'output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


In [ ]:
dummy = tokenizer("a cat",return_tensors = "tf")
_ = model(dummy)

In [ ]:
model.summary()


Model: "tfgpt2lm_head_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 transformer (TFGPT2MainLay  multiple                  27008     
 er)                                                             
                                                                 
Total params: 27008 (105.50 KB)
Trainable params: 27008 (105.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
cat_id = tokenizer.convert_tokens_to_ids("cat")
cat_before = model.transformer.wte(tf.constant([[cat_id]]))
print("cat embeddings beofre tain")
print(cat_before.numpy())

cat embeddings beofre tain
[[[ 0.017  0.014 -0.024  0.022 -0.02  -0.017  0.003  0.008 -0.015  0.02
   -0.018 -0.021  0.001  0.002 -0.001  0.017 -0.011  0.004  0.015 -0.028
    0.005 -0.022  0.004 -0.012 -0.029  0.028  0.035  0.004 -0.033  0.009
   -0.004  0.01 ]]]


In [ ]:
dataset = Dataset.from_dict({"text":sentences})

In [ ]:
def tokenize_data(bat)